# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "TODO: replace with a short sentence you want to tokenize"
print(sample_sentence)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

TODO: replace with a short sentence you want to tokenize


In [2]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    flag = "  <-- special" if token in tokenizer.all_special_tokens else ""
    print(f"{idx:>5} | {token:<12} | {token_id:>5}{flag}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101  <-- special
    1 | tod          | 28681
    2 | ##o          |  2080
    3 | :            |  1024
    4 | replace      |  5672
    5 | with         |  2007
    6 | a            |  1037
    7 | short        |  2460
    8 | sentence     |  6251
    9 | you          |  2017
   10 | want         |  2215
   11 | to           |  2000
   12 | token        | 19204
   13 | ##ize        |  4697
   14 | [SEP]        |   102  <-- special
   15 | [PAD]        |     0  <-- special
   16 | [PAD]        |     0  <-- special
   17 | [PAD]        |     0  <-- special
   18 | [PAD]        |     0  <-- special
   19 | [PAD]        |     0  <-- special
   20 | [PAD]        |     0  <-- special
   21 | [PAD]        |     0  <-- special
   22 | [PAD]        |     0  <-- special
   23 | [PAD]        |     0  <-- special

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (ind

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.



Deliverable — padding choice: Used padding="max_length" with max_length=24. The sentence is only 9 tokens (with [CLS]/[SEP]), so a fixed length of 24 gives uniform tensor shapes for batching with no truncation or lost information.

Reflection — [CLS]/[SEP]: [CLS] leads every input; its final hidden state serves as the whole-sequence summary for classification. [SEP] marks segment ends and separates sentence pairs. Both join self-attention normally.

Reflection — attention mask: Real tokens get 1, padding gets 0. Before softmax, masked positions get a large negative score, so their attention weights become ~0 and padding contributes nothing to the output.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I absolutely loved this movie, it was a fantastic experience!"
prediction = sentiment_pipeline(sentence)
prediction

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998840093612671}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?


Yes — the label matches. Words like "loved" and "fantastic" are strongly positive, so a POSITIVE prediction is expected.
The score of ~0.9998 means very high confidence. The score is the model's probability for the chosen label, so near 1.0 indicates the model is almost certain.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        text = text.strip()
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        idx = int(torch.argmax(probs))
        label = self.model.config.id2label[idx]
        return {"label": label, "probability": float(probs[idx])}

In [5]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This is the best day I have ever had, everything went perfectly!",
    "The service was terrible and I will never come back here again."
]
for text in samples:
    print(text)
    print(analyzer.predict(text))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

This is the best day I have ever had, everything went perfectly!
{'label': 'POSITIVE', 'probability': 0.9998587369918823}
The service was terrible and I will never come back here again.
{'label': 'NEGATIVE', 'probability': 0.9973881840705872}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.id2label = self.model.config.id2label

    def recognize(self, text: str):
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True
        )
        offsets = encoding.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(self.device) for k, v in encoding.items()}

        with torch.no_grad():
            logits = self.model(**inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

        entities = []
        current = None
        for token, pred_id, (start, end) in zip(tokens, pred_ids, offsets):
            label = self.id2label[pred_id]
            # skip special tokens (offset 0,0)
            if start == end == 0:
                continue
            # subword continuation: merge into current entity span
            if token.startswith("##"):
                if current is not None:
                    current["end"] = end
                continue
            if label.startswith("B-"):
                if current is not None:
                    entities.append(current)
                current = {"text": None, "entity": label[2:], "start": start, "end": end}
            elif label.startswith("I-") and current is not None and current["entity"] == label[2:]:
                current["end"] = end
            else:  # "O" or mismatched label
                if current is not None:
                    entities.append(current)
                    current = None
        if current is not None:
            entities.append(current)

        # fill in text from original string using offsets
        for ent in entities:
            ent["text"] = text[ent["start"]:ent["end"]]
        return entities

In [7]:
ner = BERTNamedEntityRecognizer()
sample_text = "Angela Merkel met Tim Cook at the Apple headquarters in California."
ner.recognize(sample_text)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Angela Merkel', 'entity': 'PER', 'start': 0, 'end': 13},
 {'text': 'Tim Cook', 'entity': 'PER', 'start': 18, 'end': 26},
 {'text': 'Apple', 'entity': 'ORG', 'start': 34, 'end': 39},
 {'text': 'California', 'entity': 'LOC', 'start': 56, 'end': 66}]

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Transformer encoder, bidirectional self-attention | Transformer decoder, causal (left-to-right) self-attention |
| Primary purpose | Understand text via masked-language modeling | Generate text via next-token prediction |
| Typical use cases | Classification, NER, extractive QA, embeddings | Text generation, chat, summarization, code completion |
| Strengths | Rich bidirectional context, strong at understanding tasks | Fluent open-ended generation, scales well, few-shot capable |
| Weaknesses | Cannot generate text naturally, needs fine-tuning per task | Unidirectional context, can hallucinate, costlier inference |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.


In [8]:
# shorter answers4:50 AMClaude responded: 1.1. Encoding: BERT (e.g. Sentence-BERT) runs text through its encoder and pools the token outputs into one fixed-length dense vector, placing semantically similar queries and documents close together in vector space.
# 2. Storage & search: Document vectors are precomputed and stored in a vector database (FAISS, Pinecone) with an approximate-nearest-neighbor index. The query is embedded the same way, and the top-k closest documents are returned by cosine/dot-product similarity.
# 3. Handoff to GPT: The top-k passages are concatenated with the question into the prompt, so GPT generates its answer grounded in that retrieved evidence instead of relying only on its parameters.
# 4. Example: A customer-support assistant—product docs are embedded and indexed, relevant snippets are retrieved per question and fed to GPT for accurate, current answers without retraining.